In [1]:
import zipfile
import os

zip_path = '/content/Fraud detection.zip'
extract_path = '/content/fraud_detection_data'

# Create the extraction directory if it doesn't exist
os.makedirs(extract_path, exist_ok=True)

# Unzip the file
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

print(f"'{zip_path}' extracted to '{extract_path}'")
print("Contents of the extracted directory:")
print(os.listdir(extract_path))

'/content/Fraud detection.zip' extracted to '/content/fraud_detection_data'
Contents of the extracted directory:
['Data', 'Src', 'Readme.md']


In [2]:
import os

data_path = os.path.join(extract_path, 'Data')
print(f"Contents of '{data_path}':")
print(os.listdir(data_path))

Contents of '/content/fraud_detection_data/Data':
['Merchant Information', 'Transaction Amounts', 'Fraudulent Patterns', 'Customer Profiles', 'Transaction Data']


In [5]:
import pandas as pd

transaction_data_path = os.path.join(data_path, 'Transaction Data', 'transaction_records.csv') # Corrected file name
try:
    df = pd.read_csv(transaction_data_path)
    print("Transaction data loaded successfully.")
    print("First 5 rows of the DataFrame:")
    print(df.head())
    print("\nDataFrame Info:")
    df.info()
except FileNotFoundError:
    print(f"Error: '{transaction_data_path}' not found. Please verify the file path and name.")
except Exception as e:
    print(f"An error occurred while loading the data: {e}")

Transaction data loaded successfully.
First 5 rows of the DataFrame:
   TransactionID     Amount  CustomerID
0              1  55.530334        1952
1              2  12.881180        1027
2              3  50.176322        1955
3              4  41.634001        1796
4              5  78.122853        1946

DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   TransactionID  1000 non-null   int64  
 1   Amount         1000 non-null   float64
 2   CustomerID     1000 non-null   int64  
dtypes: float64(1), int64(2)
memory usage: 23.6 KB


In [6]:
metadata_path = os.path.join(data_path, 'Transaction Data', 'transaction_metadata.csv')

try:
    metadata_df = pd.read_csv(metadata_path)
    print("Transaction metadata loaded successfully.")
    print("First 5 rows of the metadata DataFrame:")
    print(metadata_df.head())
    print("\nMetadata DataFrame Info:")
    metadata_df.info()

    # Merge the two dataframes
    # Assuming 'TransactionID' is the common column for merging
    df = pd.merge(df, metadata_df, on='TransactionID', how='left')
    print("\nDataFrames merged successfully. First 5 rows of the combined DataFrame:")
    print(df.head())
    print("\nCombined DataFrame Info:")
    df.info()

except FileNotFoundError:
    print(f"Error: '{metadata_path}' not found. Please verify the file path and name.")
except Exception as e:
    print(f"An error occurred while loading or merging the metadata: {e}")

Transaction metadata loaded successfully.
First 5 rows of the metadata DataFrame:
   TransactionID            Timestamp  MerchantID
0              1  2022-01-01 00:00:00        2701
1              2  2022-01-01 01:00:00        2070
2              3  2022-01-01 02:00:00        2238
3              4  2022-01-01 03:00:00        2879
4              5  2022-01-01 04:00:00        2966

Metadata DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   TransactionID  1000 non-null   int64 
 1   Timestamp      1000 non-null   object
 2   MerchantID     1000 non-null   int64 
dtypes: int64(2), object(1)
memory usage: 23.6+ KB

DataFrames merged successfully. First 5 rows of the combined DataFrame:
   TransactionID     Amount  CustomerID            Timestamp  MerchantID
0              1  55.530334        1952  2022-01-01 00:00:00        2701
1 

In [7]:
fraud_patterns_path = os.path.join(data_path, 'Fraudulent Patterns')
print(f"Contents of '{fraud_patterns_path}':")
print(os.listdir(fraud_patterns_path))

Contents of '/content/fraud_detection_data/Data/Fraudulent Patterns':
['fraud_indicators.csv', 'suspicious_activity.csv']


In [8]:
fraud_indicators_path = os.path.join(fraud_patterns_path, 'fraud_indicators.csv')

try:
    fraud_df = pd.read_csv(fraud_indicators_path)
    print("Fraud indicators loaded successfully.")
    print("First 5 rows of the fraud indicators DataFrame:")
    print(fraud_df.head())
    print("\nFraud Indicators DataFrame Info:")
    fraud_df.info()

    # Merge the fraud indicators with the main dataframe
    # Assuming 'TransactionID' is the common column and 'is_fraud' is the target variable
    df = pd.merge(df, fraud_df, on='TransactionID', how='left')
    print("\nFraud indicators merged successfully. First 5 rows of the combined DataFrame:")
    print(df.head())
    print("\nCombined DataFrame Info after merging fraud indicators:")
    df.info()

    # Fill any potential NaN values in 'is_fraud' if some transactions had no indicator (assume not fraudulent)
    if 'is_fraud' in df.columns:
        df['is_fraud'] = df['is_fraud'].fillna(0).astype(int)
        print("\nDistribution of 'is_fraud' column after merging and filling NaNs:")
        print(df['is_fraud'].value_counts())
    else:
        print("Warning: 'is_fraud' column not found in fraud_indicators.csv. Please check the file content.")

except FileNotFoundError:
    print(f"Error: '{fraud_indicators_path}' not found. Please verify the file path and name.")
except Exception as e:
    print(f"An error occurred while loading or merging the fraud indicators: {e}")

Fraud indicators loaded successfully.
First 5 rows of the fraud indicators DataFrame:
   TransactionID  FraudIndicator
0              1               0
1              2               0
2              3               0
3              4               0
4              5               0

Fraud Indicators DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 2 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   TransactionID   1000 non-null   int64
 1   FraudIndicator  1000 non-null   int64
dtypes: int64(2)
memory usage: 15.8 KB

Fraud indicators merged successfully. First 5 rows of the combined DataFrame:
   TransactionID     Amount  CustomerID            Timestamp  MerchantID  \
0              1  55.530334        1952  2022-01-01 00:00:00        2701   
1              2  12.881180        1027  2022-01-01 01:00:00        2070   
2              3  50.176322        1955  2022-01-01 02:00:0

In [9]:
if 'FraudIndicator' in df.columns:
    df = df.rename(columns={'FraudIndicator': 'is_fraud'})
    print("Column 'FraudIndicator' renamed to 'is_fraud'.")

print("\nDistribution of the target variable 'is_fraud':")
print(df['is_fraud'].value_counts())
print("\nPercentage of fraudulent transactions:")
print(df['is_fraud'].value_counts(normalize=True) * 100)

Column 'FraudIndicator' renamed to 'is_fraud'.

Distribution of the target variable 'is_fraud':
is_fraud
0    955
1     45
Name: count, dtype: int64

Percentage of fraudulent transactions:
is_fraud
0    95.5
1     4.5
Name: proportion, dtype: float64


In [10]:
print("Preprocessing data...")

# Convert Timestamp to datetime
df['Timestamp'] = pd.to_datetime(df['Timestamp'])

# Extract time-based features
df['Hour'] = df['Timestamp'].dt.hour
df['DayOfWeek'] = df['Timestamp'].dt.dayofweek
df['Month'] = df['Timestamp'].dt.month

# Drop original Timestamp and TransactionID (as it's an identifier)
df = df.drop(columns=['Timestamp', 'TransactionID'])

print("Data preprocessing complete. First 5 rows of the processed DataFrame:")
print(df.head())
print("\nProcessed DataFrame Info:")
df.info()

Preprocessing data...
Data preprocessing complete. First 5 rows of the processed DataFrame:
      Amount  CustomerID  MerchantID  is_fraud  Hour  DayOfWeek  Month
0  55.530334        1952        2701         0     0          5      1
1  12.881180        1027        2070         0     1          5      1
2  50.176322        1955        2238         0     2          5      1
3  41.634001        1796        2879         0     3          5      1
4  78.122853        1946        2966         0     4          5      1

Processed DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Amount      1000 non-null   float64
 1   CustomerID  1000 non-null   int64  
 2   MerchantID  1000 non-null   int64  
 3   is_fraud    1000 non-null   int64  
 4   Hour        1000 non-null   int32  
 5   DayOfWeek   1000 non-null   int32  
 6   Month       1000 

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

print("Preparing data for modeling...")

# Separate features (X) and target (y)
X = df.drop('is_fraud', axis=1)
y = df['is_fraud']

# Drop high-cardinality identifier columns for now
# In a real-world scenario, these might be encoded differently or used in more advanced models
X = X.drop(columns=['CustomerID', 'MerchantID'])

print("Features (X) and Target (y) separated. Identifier columns dropped.")
print(f"X shape: {X.shape}, y shape: {y.shape}")

# Split data into training and testing sets
# Using stratify=y to maintain the proportion of fraudulent transactions in both sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print(f"\nTraining set shape: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"Testing set shape: X_test={X_test.shape}, y_test={y_test.shape}")
print(f"Distribution of target in training set:\n{y_train.value_counts(normalize=True)}")
print(f"Distribution of target in testing set:\n{y_test.value_counts(normalize=True)}")

# Feature Scaling (numerical features)
# Identify numerical columns to scale
numerical_cols = ['Amount', 'Hour', 'DayOfWeek', 'Month']

scaler = StandardScaler()
X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

print("\nNumerical features scaled. First 5 rows of scaled X_train:")
print(X_train.head())
print("Data preparation complete for model training.")

Preparing data for modeling...
Features (X) and Target (y) separated. Identifier columns dropped.
X shape: (1000, 4), y shape: (1000,)

Training set shape: X_train=(800, 4), y_train=(800,)
Testing set shape: X_test=(200, 4), y_test=(200,)
Distribution of target in training set:
is_fraud
0    0.955
1    0.045
Name: proportion, dtype: float64
Distribution of target in testing set:
is_fraud
0    0.955
1    0.045
Name: proportion, dtype: float64

Numerical features scaled. First 5 rows of scaled X_train:
       Amount      Hour  DayOfWeek     Month
621  0.868395  1.422038  -0.505322 -0.588898
184 -0.994887  0.691382   0.995635 -0.588898
233  0.547359  0.837513  -1.505960 -0.588898
226  0.659199 -0.185404  -1.505960 -0.588898
562 -0.187247 -0.185404  -1.505960 -0.588898
Data preparation complete for model training.


In [12]:
from imblearn.over_sampling import SMOTE

print("Applying SMOTE to the training data...")

smt = SMOTE(random_state=42)
X_train_smote, y_train_smote = smt.fit_resample(X_train, y_train)

print("SMOTE application complete.")
print(f"Original training set shape: {X_train.shape}")
print(f"SMOTE training set shape: {X_train_smote.shape}")
print("\nOriginal training target distribution:")
print(y_train.value_counts())
print("\nSMOTE training target distribution:")
print(y_train_smote.value_counts())

Applying SMOTE to the training data...
SMOTE application complete.
Original training set shape: (800, 4)
SMOTE training set shape: (1528, 4)

Original training target distribution:
is_fraud
0    764
1     36
Name: count, dtype: int64

SMOTE training target distribution:
is_fraud
0    764
1    764
Name: count, dtype: int64


In [13]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, roc_auc_score, confusion_matrix, classification_report

# Function to evaluate model performance
def evaluate_model(model, X_test, y_test, model_name="Model"):
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    print(f"\n--- {model_name} Performance ---")
    print(f"Classification Report:\n{classification_report(y_test, y_pred)}")
    print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")

    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_proba) if y_proba is not None else "N/A"

    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"ROC-AUC: {roc_auc:.4f}")

    return {'precision': precision, 'recall': recall, 'roc_auc': roc_auc}

print("Training Logistic Regression model...")
# Logistic Regression
log_reg = LogisticRegression(random_state=42, solver='liblinear') # 'liblinear' is good for small datasets and L1/L2 regularization
log_reg.fit(X_train_smote, y_train_smote)

print("Logistic Regression training complete. Evaluating...")
log_reg_metrics = evaluate_model(log_reg, X_test, y_test, "Logistic Regression")

print("\nTraining Random Forest model...")
# Random Forest
# Using class_weight='balanced' could be an alternative to SMOTE, but since we already applied SMOTE, we'll train on the balanced data.
rf_clf = RandomForestClassifier(random_state=42, n_estimators=100)
rf_clf.fit(X_train_smote, y_train_smote)

print("Random Forest training complete. Evaluating...")
rf_metrics = evaluate_model(rf_clf, X_test, y_test, "Random Forest")

print("\nModel training and initial evaluation complete.")

Training Logistic Regression model...
Logistic Regression training complete. Evaluating...

--- Logistic Regression Performance ---
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.45      0.61       191
           1       0.05      0.56      0.08         9

    accuracy                           0.46       200
   macro avg       0.50      0.50      0.35       200
weighted avg       0.91      0.46      0.59       200

Confusion Matrix:
[[ 86 105]
 [  4   5]]
Precision: 0.0455
Recall: 0.5556
ROC-AUC: 0.4747

Training Random Forest model...
Random Forest training complete. Evaluating...

--- Random Forest Performance ---
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95       191
           1       0.09      0.11      0.10         9

    accuracy                           0.91       200
   macro avg       0.52      0.53      0.53       200
weighted avg      

In [14]:
from sklearn.model_selection import RandomizedSearchCV

print("Performing hyperparameter tuning for Random Forest using RandomizedSearchCV...")

# Define the parameter distribution for RandomizedSearchCV
# n_estimators: number of trees in the forest
# max_features: number of features to consider when looking for the best split
# max_depth: maximum number of levels in tree
# min_samples_split: minimum number of samples required to split an internal node
# min_samples_leaf: minimum number of samples required to be at a leaf node
# bootstrap: whether bootstrap samples are used when building trees
param_dist = {
    'n_estimators': [100, 200, 300, 400, 500],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [10, 20, 30, 40, 50, None], # None means nodes are expanded until all leaves are pure or until all leaves contain less than min_samples_split samples.
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'bootstrap': [True, False]
}

# Instantiate the Random Forest classifier
rf_base = RandomForestClassifier(random_state=42)

# Setup RandomizedSearchCV
# n_iter: number of parameter settings that are sampled (trade-off between runtime and quality of the solution)
# scoring: metric to evaluate the models (e.g., 'roc_auc' is good for imbalanced datasets)
# cv: number of cross-validation folds
# verbose: controls the verbosity (higher value means more messages)
random_search_rf = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_dist,
    n_iter=20, # Reduced for quicker execution, can be increased for more thorough search
    scoring='roc_auc', # Focus on ROC-AUC for imbalanced data
    cv=5,
    verbose=2,
    random_state=42,
    n_jobs=-1 # Use all available cores
)

# Fit RandomizedSearchCV to the SMOTE-resampled training data
random_search_rf.fit(X_train_smote, y_train_smote)

print("\nRandom Forest Hyperparameter Tuning Complete.")
print(f"Best parameters found: {random_search_rf.best_params_}")
print(f"Best ROC-AUC score on training set: {random_search_rf.best_score_:.4f}")

# Get the best model
best_rf_model = random_search_rf.best_estimator_

print("\nEvaluating the best Random Forest model on the test set...")
best_rf_metrics = evaluate_model(best_rf_model, X_test, y_test, "Optimized Random Forest")

Performing hyperparameter tuning for Random Forest using RandomizedSearchCV...
Fitting 5 folds for each of 20 candidates, totalling 100 fits

Random Forest Hyperparameter Tuning Complete.
Best parameters found: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2', 'max_depth': None, 'bootstrap': True}
Best ROC-AUC score on training set: 0.9764

Evaluating the best Random Forest model on the test set...

--- Optimized Random Forest Performance ---
Classification Report:
              precision    recall  f1-score   support

           0       0.96      0.95      0.95       191
           1       0.09      0.11      0.10         9

    accuracy                           0.91       200
   macro avg       0.52      0.53      0.53       200
weighted avg       0.92      0.91      0.91       200

Confusion Matrix:
[[181  10]
 [  8   1]]
Precision: 0.0909
Recall: 0.1111
ROC-AUC: 0.5782


In [4]:
transaction_dir_path = os.path.join(data_path, 'Transaction Data')
print(f"Contents of '{transaction_dir_path}':")
print(os.listdir(transaction_dir_path))

Contents of '/content/fraud_detection_data/Data/Transaction Data':
['transaction_metadata.csv', 'transaction_records.csv']
